# U4A3. Modelo introductorio para el gemelo digital

**Problema seleccionado:** clasificación multiclase del estado operativo del robot seguidor de línea.

**Trazabilidad del dataset:** `dataset/v0.2/datos_limpios.csv`, captura física del 23 de abril de 2026. La versión limpia conserva 322 muestras de dos sesiones. Las clases son `straight`, `curve` y `recovery`.

Se utilizan únicamente variables sensoriales disponibles para el flujo del gemelo digital: lecturas normalizadas de los cuatro sensores, posición estimada de línea y balance izquierda-derecha. Se omiten columnas derivadas del etiquetado para reducir fuga de información.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent

DATASET_PATH = ROOT / "dataset" / "v0.2" / "datos_limpios.csv"
RESULTS_DIR = ROOT / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "dataset_label"
FEATURES = ["norm0", "norm1", "norm2", "norm3", "line_pos", "balance_lr"]
CLASS_ORDER = ["straight", "curve", "recovery"]
RANDOM_STATE = 42
TEST_SIZE = 0.20


## Carga y revisión mínima de datos


In [ ]:
df = pd.read_csv(DATASET_PATH)
required = [TARGET, "dataset_session", *FEATURES]
clean = df.dropna(subset=required).copy()

print(f"Muestras utilizadas: {len(clean)}")
print("\nClases:")
display(clean[TARGET].value_counts().rename_axis("clase").to_frame("muestras"))
print("\nSesiones:")
display(clean["dataset_session"].value_counts().rename_axis("sesion").to_frame("muestras"))


In [ ]:
counts = clean[TARGET].value_counts().reindex(CLASS_ORDER).fillna(0)
ax = counts.plot(kind="bar", color=["#2980B9", "#16A085", "#D35400"], figsize=(7, 4), rot=0)
ax.set_title("Distribución de clases del dataset v0.2")
ax.set_xlabel("Estado del robot")
ax.set_ylabel("Número de muestras")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Línea base y modelo

La línea base predice siempre la clase mayoritaria. El modelo introductorio es KNN con `k=5`, precedido por estandarización porque las variables usan escalas distintas. La partición es estratificada: 80 % entrenamiento y 20 % prueba, con semilla fija `42`.


In [ ]:
X = clean[FEATURES]
y = clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
pred_knn = model.predict(X_test)

metrics = pd.DataFrame([
    {
        "modelo": "Baseline: clase mayoritaria",
        "accuracy": accuracy_score(y_test, pred_baseline),
        "f1_macro": f1_score(y_test, pred_baseline, average="macro"),
        "observación técnica": "Referencia mínima: predice siempre curve.",
    },
    {
        "modelo": "KNN k=5",
        "accuracy": accuracy_score(y_test, pred_knn),
        "f1_macro": f1_score(y_test, pred_knn, average="macro"),
        "observación técnica": "Mejora ambas métricas usando telemetría sensorial.",
    },
]).round({"accuracy": 4, "f1_macro": 4})
display(metrics)


## Matriz de confusión y reporte por clase


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_knn, labels=CLASS_ORDER, display_labels=CLASS_ORDER,
    cmap="Blues", colorbar=False, values_format="d", ax=ax
)
ax.set_title("Matriz de confusión - KNN k=5")
plt.tight_layout()
plt.show()

display(pd.DataFrame(classification_report(
    y_test, pred_knn, labels=CLASS_ORDER, output_dict=True, zero_division=0
)).T.round(4))


## Variables con mayor influencia


In [ ]:
importance = permutation_importance(
    model, X_test, y_test, scoring="f1_macro", n_repeats=30, random_state=RANDOM_STATE
)
importance_df = pd.DataFrame({
    "variable": FEATURES,
    "importancia_media": importance.importances_mean,
    "desviacion": importance.importances_std,
}).sort_values("importancia_media", ascending=False)
display(importance_df.round(4))

ax = importance_df.sort_values("importancia_media").plot(
    kind="barh", x="variable", y="importancia_media", xerr="desviacion",
    color="#16A085", legend=False, figsize=(7, 4.5)
)
ax.set_title("Importancia por permutación")
ax.set_xlabel("Reducción media del F1-score macro")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## Validación suplementaria por sesión


In [ ]:
rows = []
for held_session in sorted(clean["dataset_session"].unique()):
    train = clean[clean["dataset_session"] != held_session]
    test = clean[clean["dataset_session"] == held_session]
    session_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    session_model.fit(train[FEATURES], train[TARGET])
    pred = session_model.predict(test[FEATURES])
    rows.append({
        "sesión reservada": held_session,
        "muestras de prueba": len(test),
        "accuracy": accuracy_score(test[TARGET], pred),
        "f1_macro": f1_score(test[TARGET], pred, average="macro"),
    })
session_validation = pd.DataFrame(rows).round(4)
display(session_validation)


## Interpretación técnica

- **¿El modelo mejora al baseline?** Sí. En la prueba estratificada, KNN incrementa `Accuracy` de `0.6923` a `0.8462` y el `F1-score macro` de `0.2727` a `0.7771`.
- **¿Qué variables parecen influir más?** Las lecturas normalizadas `norm3`, `norm2`, `norm1` y `norm0` presentan la mayor influencia por permutación. Esto coincide con el comportamiento esperado de un seguidor de línea.
- **¿Es suficiente para continuar?** Sí, como modelo introductorio. Puede integrarse al gemelo digital para estimar estados operativos y comparar el comportamiento simulado contra el real.
- **¿Qué limitaciones existen?** La versión `v0.2` solo contiene 322 muestras de dos sesiones y mantiene desbalance hacia la clase `curve`. Conviene capturar más recorridos antes de considerar un modelo definitivo.
